# Neuro-Symbolic MLP — Twitch Dataset

MLP with optional symbolic regularization loss on the enriched 9-feature dataset.

In [52]:
# ── Experiment Configuration ─────────────────────────────────────────
# Toggle before running. Controls loss function AND output directory.

USE_SYMBOLIC_LOSS = True   # False → plain MLP baseline
LAMBDA            = 2.0    # symbolic loss weight (used only if USE_SYMBOLIC_LOSS=True)
PA_THRESHOLD      = 30     # min preferential attachment to trigger rule
MARGIN            = 0.4    # hinge margin
# ─────────────────────────────────────────────────────────────────────


In [53]:
import os
from pathlib import Path

# Find project root regardless of VS Code working directory
_cwd = Path(os.getcwd())
PROJECT_ROOT = next(
    p for p in [_cwd] + list(_cwd.parents)
    if (p / "data" / "twitch.csv").exists()
)

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report
)

RANDOM_SEED      = 42
TARGET_COLUMN    = "link"
FEATURE_COLUMNS  = [
    "common_neighbors", "jaccard_coefficient", "adamic_adar", "preferential_attachment",
    "views_diff", "views_ratio", "age_diff", "same_partner", "same_mature"
]
# AA=index 2, PA=index 3 — used by symbolic loss mask
AA_IDX = 2
PA_IDX = 3

torch.manual_seed(RANDOM_SEED)

df = pd.read_csv(str(PROJECT_ROOT / "data" / "twitch.csv"))
print(f"Rows: {len(df):,}  Columns: {list(df.columns)}")


Rows: 70,648  Columns: ['from', 'to', 'common_neighbors', 'jaccard_coefficient', 'adamic_adar', 'preferential_attachment', 'link', 'views_diff', 'views_ratio', 'age_diff', 'same_partner', 'same_mature']


## Train / Test Split

In [54]:
X = df[FEATURE_COLUMNS].values.astype(float)
y = df[TARGET_COLUMN].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Training set: {len(X_train):,}")
print(f"Test set:     {len(X_test):,}")


Training set: 56,518
Test set:     14,130


## MLP Architecture

In [55]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


## Training Configuration

In [56]:
EPOCHS            = 50
BATCH_SIZE        = 256
LR                = 1e-3

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    for X_batch, X_raw_batch, y_batch in loader:
        optimizer.zero_grad()
        probs    = model(X_batch)
        bce_loss = criterion(probs, y_batch)

        aa   = X_raw_batch[:, AA_IDX]
        pa   = X_raw_batch[:, PA_IDX]
        mask = (y_batch == 1) & (aa < 1e-6) & (pa > PA_THRESHOLD)

        if USE_SYMBOLIC_LOSS and mask.sum() > 0:
            sym_loss = torch.clamp(MARGIN - probs[mask], min=0).mean()
            loss = bce_loss + LAMBDA * sym_loss
        else:
            loss = bce_loss

        loss.backward()
        optimizer.step()

@torch.no_grad()
def predict(model, X_tensor):
    model.eval()
    probs = model(X_tensor).numpy()
    preds = (probs >= 0.5).astype(int)
    return probs, preds


## 10-Fold Cross-Validation

In [57]:
X_train_raw = scaler.inverse_transform(X_train)

kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)
cv_scores = {"AUC": [], "Accuracy": [], "Weighted Precision": [],
             "Weighted Recall": [], "Weighted F1": []}

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
    Xtr     = torch.tensor(X_train[train_idx],     dtype=torch.float32)
    Xtr_raw = torch.tensor(X_train_raw[train_idx], dtype=torch.float32)
    ytr     = torch.tensor(y_train[train_idx],     dtype=torch.float32)
    Xval    = torch.tensor(X_train[val_idx],       dtype=torch.float32)
    yval    = y_train[val_idx]

    loader    = DataLoader(TensorDataset(Xtr, Xtr_raw, ytr), batch_size=BATCH_SIZE, shuffle=True)
    model     = MLP(input_dim=len(FEATURE_COLUMNS))
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.BCELoss()

    for epoch in range(EPOCHS):
        train_epoch(model, loader, optimizer, criterion)

    probs, preds = predict(model, Xval)
    cv_scores["AUC"].append(roc_auc_score(yval, probs))
    cv_scores["Accuracy"].append(accuracy_score(yval, preds))
    cv_scores["Weighted Precision"].append(precision_score(yval, preds, average="weighted", zero_division=0))
    cv_scores["Weighted Recall"].append(recall_score(yval, preds, average="weighted"))
    cv_scores["Weighted F1"].append(f1_score(yval, preds, average="weighted"))
    print(f"Fold {fold+1}  Acc: {cv_scores['Accuracy'][-1]:.3f}  AUC: {cv_scores['AUC'][-1]:.3f}")

print()
print("Average CV Metrics:")
for name, scores in cv_scores.items():
    print(f"  {name:<25}: {np.mean(scores):.3f} (+/- {np.std(scores):.3f})")


Fold 1  Acc: 0.865  AUC: 0.933
Fold 2  Acc: 0.853  AUC: 0.928
Fold 3  Acc: 0.844  AUC: 0.921
Fold 4  Acc: 0.850  AUC: 0.921
Fold 5  Acc: 0.858  AUC: 0.926
Fold 6  Acc: 0.852  AUC: 0.924
Fold 7  Acc: 0.860  AUC: 0.922
Fold 8  Acc: 0.849  AUC: 0.920
Fold 9  Acc: 0.852  AUC: 0.921
Fold 10  Acc: 0.849  AUC: 0.919

Average CV Metrics:
  AUC                      : 0.923 (+/- 0.004)
  Accuracy                 : 0.853 (+/- 0.006)
  Weighted Precision       : 0.855 (+/- 0.007)
  Weighted Recall          : 0.853 (+/- 0.006)
  Weighted F1              : 0.853 (+/- 0.006)


## Test Set Evaluation

In [58]:
X_train_t     = torch.tensor(X_train,                          dtype=torch.float32)
y_train_t     = torch.tensor(y_train,                          dtype=torch.float32)
X_test_t      = torch.tensor(X_test,                           dtype=torch.float32)
X_train_raw_t = torch.tensor(scaler.inverse_transform(X_train), dtype=torch.float32)

loader    = DataLoader(TensorDataset(X_train_t, X_train_raw_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
model     = MLP(input_dim=len(FEATURE_COLUMNS))
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCELoss()

for epoch in range(EPOCHS):
    train_epoch(model, loader, optimizer, criterion)

probs_test, preds_test = predict(model, X_test_t)

print(f"AUC:      {roc_auc_score(y_test, probs_test):.3f}")
print(f"Accuracy: {accuracy_score(y_test, preds_test):.3f}")
print()
print(classification_report(y_test, preds_test, target_names=["Non-Link", "Link"]))


AUC:      0.923
Accuracy: 0.854

              precision    recall  f1-score   support

    Non-Link       0.83      0.89      0.86      7065
        Link       0.88      0.82      0.85      7065

    accuracy                           0.85     14130
   macro avg       0.86      0.85      0.85     14130
weighted avg       0.86      0.85      0.85     14130



## Error Analysis & Save Outputs

In [59]:
import os
from datetime import datetime

# ── Reconstruct test set metadata ─────────────────────────────────────────
indices = np.arange(len(df))
idx_train, idx_test = train_test_split(
    indices, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)
df_test_meta = df.iloc[idx_test][["from", "to"] + FEATURE_COLUMNS].reset_index(drop=True)
df_results = df_test_meta.copy()
df_results["link"]        = y_test
df_results["prediction"]  = preds_test
df_results["probability"] = probs_test

fn = df_results[(df_results["link"] == 1) & (df_results["prediction"] == 0)]
fp = df_results[(df_results["link"] == 0) & (df_results["prediction"] == 1)]

print(f"False negatives: {len(fn)}")
print(f"False positives: {len(fp)}")
print(f"FN: AA=0 rate:   {(fn.adamic_adar == 0).mean()*100:.1f}%")
print(f"FN: PA median:   {fn.preferential_attachment.median():.0f}")

# ── Save outputs ──────────────────────────────────────────────────────────
ts = datetime.now().strftime("%Y%m%d_%H%M")
if USE_SYMBOLIC_LOSS:
    out_dir = PROJECT_ROOT / "notebooks" / "output" / f"{ts}_mlp_lambda{LAMBDA}"
else:
    out_dir = PROJECT_ROOT / "notebooks" / "output" / f"{ts}_mlp_bce"

os.makedirs(out_dir, exist_ok=True)
cfg = f"lambda{LAMBDA}" if USE_SYMBOLIC_LOSS else "bce"
fn.to_csv(out_dir / f"mlp_full_{cfg}_fn.csv", index=False)
fp.to_csv(out_dir / f"mlp_full_{cfg}_fp.csv", index=False)
print(f"Saved to {out_dir.relative_to(PROJECT_ROOT)}")


False negatives: 1263
False positives: 806
FN: AA=0 rate:   94.7%
FN: PA median:   54
